# Experiment: DDPM Re-evaluation on v2.3 Data

**Date:** 2026-03-07  
**Experiment ID:** `ddpm_v23_seed42`  
**Status:** Preliminary (1-seed scout)  
**Type:** Training  
**GitHub Issue:** [#49](https://github.com/wrockey/vmat-diffusion/issues/49)  

---

## 1. Overview

### 1.1 Objective

Re-evaluate the DDPM (Denoising Diffusion Probabilistic Model) on the corrected v2.3 dataset (n=74). The DDPM was previously dismissed based on corrupted v2.2.0 data (n=23, D95 artifact). This scout tests whether diffusion-based dose prediction improves global/low-dose gamma compared to direct regression (baseline U-Net), while maintaining PTV quality.

**Hypothesis:** DDPM's iterative refinement may produce smoother, more physically realistic dose distributions in low-dose regions, potentially improving global gamma pass rate where the baseline struggles (~30%).

**Comparison framing:** DDPM (MSE + physics penalty) is most fairly compared to **baseline_v23** (MSE-only). The combined_loss_2.5:1 is included as "current best" ceiling but uses loss engineering the DDPM does not have.

### 1.2 Key Results

| Metric | DDPM (seed42) | Baseline (seed42, MSE) | Combined 2.5:1 (seed42) |
|--------|--------------|------------------------|-------------------------|
| MAE (Gy) | 9.04 ± 1.95 | 4.80 ± 2.45 | 4.88 ± 2.09 |
| Global Gamma 3%/3mm | 9.2 ± 2.1% | 28.1 ± 12.6% | 27.4 ± 11.2% |
| PTV Gamma 3%/3mm | 0.0% (all cases) | 85.5 ± 10.9% | 95.1 ± 3.5% |
| PTV70 D95 Gap (Gy) | -65.5 ± 0.8 | -0.86 ± 0.92 | +0.12 ± 0.60 |
| PTV70 D95 predicted | 4.0-6.1 Gy | ~70 Gy | ~71 Gy |
| Training time | 33.3 h (107 epochs) | ~6 h (200 epochs) | ~6 h (200 epochs) |

### 1.3 Conclusion

**The DDPM fails catastrophically on v2.3 data.** It predicts only ~10-15% of the correct dose magnitude in the PTV (D95 of 4-6 Gy vs target ~70 Gy), resulting in 0% PTV gamma pass rate and 9.2% global gamma — dramatically worse than the MSE-only baseline on every metric. The denoising process produces spatially diffuse, low-magnitude dose distributions that bear no clinical resemblance to the ground truth. This definitively closes the DDPM investigation for this project.

---

## 2. What Changed

Compared to **baseline_v23 (seed42)** (`82bddc5`), this experiment changes:

| Parameter | Baseline | DDPM |
|-----------|---------|------|
| Model class | BaselineUNet3D (direct regression) | DoseDDPM (SimpleUNet3D backbone) |
| Loss function | MSE | MSE on noise prediction + negative dose penalty |
| Training paradigm | Direct dose prediction | Epsilon (noise) prediction with cosine schedule |
| Inference | Single forward pass | DDIM 50-step iterative denoising |
| Input channels | 9 (CT + 8 SDFs) | 10 (noisy dose + CT + 8 SDFs) |
| Conditioning | FiLM embedding of constraints | FiLM embedding of constraints |
| Timesteps | N/A | 1000 (cosine schedule) |
| Early stopping patience | 50 | 50 (fixed from 20 for this run) |

**Architecture is comparable:** Both use ~23.7M parameters, base_channels=48, same U-Net depth. The key difference is the training paradigm (direct regression vs diffusion).

---

## 3. Reproducibility

In [ ]:
import json
from pathlib import Path

REPRODUCIBILITY = {
    'git_commit': 'adfb840',
    'git_message': 'fix: Clamp DDPM patch center range for small volumes',
    'python_version': '3.12.12',
    'pytorch_version': '2.10.0+cu126',
    'cuda_version': '12.6',
    'gpu': 'NVIDIA GeForce RTX 3090',
    'random_seed': 42,
    'experiment_date': '2026-03-05',
}

print('Reproducibility Information:')
for k, v in REPRODUCIBILITY.items():
    print(f'  {k}: {v}')

EXP_NAME = 'ddpm_v23_seed42'
env_snapshot = Path(f'../runs/{EXP_NAME}/environment_snapshot.txt')
print(f'\n  Environment snapshot: {env_snapshot} ({"EXISTS" if env_snapshot.exists() else "MISSING"})')

### Command to Reproduce

```bash
# 1. Checkout exact code
git checkout adfb840

# 2. Activate environment
conda activate vmat-diffusion

# 3. Train DDPM
python scripts/train_dose_ddpm_v2.py \
    --data_dir /home/wrockey/data/processed_npz \
    --exp_name ddpm_v23_seed42 \
    --epochs 200 --seed 42 \
    --base_channels 48 \
    --schedule cosine --timesteps 1000 \
    --batch_size 2 --lr 0.0001 --num_workers 2

# 4. Evaluate (checkpoint has / in metric name — use find)
BEST_CKPT=$(find runs/ddpm_v23_seed42/checkpoints -name '*.ckpt' -not -name 'last*' | sort -t= -k3 -n | head -1)
python scripts/inference_dose_ddpm.py \
    --checkpoint "$BEST_CKPT" \
    --input_dir /home/wrockey/data/processed_npz_test \
    --output_dir predictions/ddpm_v23_seed42_test \
    --compute_metrics --overlap 64 --gamma_subsample 4 \
    --ddim_steps 50
```

---

## 4. Dataset

In [ ]:
import json
from pathlib import Path

EXP_NAME = 'ddpm_v23_seed42'

# Load test cases
tc = json.load(open(f'../runs/{EXP_NAME}/test_cases.json'))
config = json.load(open(f'../runs/{EXP_NAME}/training_config.json'))

DATASET = {
    'preprocessing_version': 'v2.3.0',
    'total_cases': 74,
    'train_cases': config['train_cases'],
    'val_cases': config['val_cases'],
    'test_case_ids': sorted(tc['test_cases']),
    'train_case_ids': sorted(config['train_case_ids']),
    'val_case_ids': sorted(config['val_case_ids']),
}

print(f'Preprocessing: {DATASET["preprocessing_version"]}')
print(f'Total cases: {DATASET["total_cases"]}')
print(f'Split: {DATASET["train_cases"]} train / {DATASET["val_cases"]} val / {len(DATASET["test_case_ids"])} test')
print(f'\nTest cases: {DATASET["test_case_ids"]}')
print(f'Val cases: {DATASET["val_case_ids"]}')

---

## 5. Model & Training Configuration

In [ ]:
import json
from pathlib import Path

EXP_NAME = 'ddpm_v23_seed42'
config = json.load(open(f'../runs/{EXP_NAME}/training_config.json'))
summary = json.load(open(f'../runs/{EXP_NAME}/training_summary.json'))

print('=== Model ===')
print(f'  Class: DoseDDPM (backbone: {config["model_class"]})')
print(f'  Parameters: {config["total_parameters"]:,}')
print(f'  GPU: {config["gpu_name"]}')
print()

print('=== Hyperparameters ===')
for k, v in sorted(config['hyperparameters'].items()):
    print(f'  {k}: {v}')
print()

print('=== Training ===')
print(f'  Max epochs: {config["max_epochs"]}')
print(f'  Actual epochs: {summary["total_epochs_completed"]}')
print(f'  Early stopped: {summary["early_stopped"]}')
print(f'  Best epoch: {summary["best_epoch"]}')
print(f'  Best val MAE: {summary["best_val_mae_gy"]:.2f} Gy')
print(f'  Training time: {summary["total_training_time_hours"]:.1f} hours')
print(f'  Avg epoch time: {summary["avg_epoch_time_sec"]:.0f} sec ({summary["avg_epoch_time_sec"]/60:.1f} min)')
print(f'  Precision: {config["precision"]}')

---

## 6. Results

Figures generated by `scripts/generate_ddpm_v23_figures.py` and saved to `runs/ddpm_v23/figures/`.

### 6.1 Training Curves

![Training Curves](../runs/ddpm_v23/figures/fig1_training_curves.png)

**Caption:** Training loss and validation MAE vs epoch for DDPM v2.3 (seed 42). The model trained for 107 epochs before early stopping (patience=50, best at epoch 55). Validation MAE is extremely noisy (5-84 Gy range), characteristic of DDIM-based validation where the stochastic denoising process produces highly variable single-patch predictions. Best val MAE was 5.62 Gy (vs baseline's ~3.5 Gy).

**Key observations:**
- Val MAE never stabilized — oscillated wildly throughout training
- Training loss converged smoothly, suggesting the noise prediction objective was learned
- The disconnect between stable training loss and erratic val MAE suggests the denoising/sampling process is the bottleneck, not the noise predictor

### 6.2 Dose Colorwash (Representative Case: prostate70gy_0027)

![Dose Colorwash](../runs/ddpm_v23/figures/fig2_dose_colorwash.png)

**Caption:** Predicted (top) vs ground truth (bottom) dose distribution overlaid on CT for prostate70gy_0027. Axial slice through PTV70 centroid. The DDPM prediction shows spatially diffuse, low-magnitude dose with no concentrated high-dose region in the PTV. Ground truth shows the expected ~70 Gy hotspot; the DDPM produces only ~5-10 Gy in the same region.

### 6.3 Dose Difference Map

![Dose Difference](../runs/ddpm_v23/figures/fig3_dose_difference.png)

**Caption:** Dose difference (predicted minus ground truth) for prostate70gy_0027. The massive blue (underdose) region centered on the PTV confirms the DDPM predicts ~60 Gy less than ground truth in the target volume. Errors are not random — the model systematically fails to concentrate dose in the PTV.

### 6.4 DVH Comparison

![DVH](../runs/ddpm_v23/figures/fig4_dvh_comparison.png)

**Caption:** DVH curves for predicted (dashed) vs ground truth (solid) for all structures, prostate70gy_0027. The predicted PTV70 DVH is shifted ~60 Gy to the left of ground truth, showing the model predicts essentially no therapeutic dose in the target. OAR DVHs are also poorly matched. No clinical constraints are met.

### 6.5 Gamma Analysis

![Gamma](../runs/ddpm_v23/figures/fig5_gamma_bar_chart.png)

**Caption:** Global gamma 3%/3mm pass rate across 7 test cases. All cases are below 15%, with a mean of 9.2%. This is dramatically worse than the baseline (28.1%) and combined loss (27.4%). PTV gamma is 0% for all cases — no voxels in the PTV pass the 3%/3mm criterion.

### 6.6 Per-Case Results

![Box Plots](../runs/ddpm_v23/figures/fig6_per_case_boxplots.png)

**Caption:** Distribution of MAE and gamma pass rate across 7 test cases. MAE ranges 6.7-12.1 Gy (baseline: 2.1-8.8 Gy). The failure is consistent across all cases — not driven by outliers.

---

## 7. Statistical Analysis

In [ ]:
import json
import numpy as np
from scipy import stats

# Load DDPM results
ddpm = json.load(open('../predictions/ddpm_v23_seed42_test/ddpm_evaluation_results.json'))
baseline = json.load(open('../predictions/baseline_v23_seed42_test/baseline_evaluation_results.json'))

# Extract per-case metrics
def extract_metrics(results):
    mae = [r['dose_metrics']['mae_gy'] for r in results['per_case_results']]
    gamma = [r['gamma']['global_3mm3pct']['gamma_pass_rate'] for r in results['per_case_results']]
    d95 = [r['dvh_metrics']['PTV70']['D95_error'] for r in results['per_case_results']]
    return np.array(mae), np.array(gamma), np.array(d95)

ddpm_mae, ddpm_gamma, ddpm_d95 = extract_metrics(ddpm)
base_mae, base_gamma, base_d95 = extract_metrics(baseline)

print('=== DDPM vs Baseline (seed42, paired, n=7) ===')
print()

for name, ddpm_vals, base_vals in [
    ('MAE (Gy)', ddpm_mae, base_mae),
    ('Global Gamma (%)', ddpm_gamma, base_gamma),
    ('PTV70 D95 Gap (Gy)', ddpm_d95, base_d95),
]:
    diff = ddpm_vals - base_vals
    stat, p = stats.wilcoxon(ddpm_vals, base_vals)
    print(f'{name}:')
    print(f'  DDPM:     {np.mean(ddpm_vals):.2f} +/- {np.std(ddpm_vals):.2f}')
    print(f'  Baseline: {np.mean(base_vals):.2f} +/- {np.std(base_vals):.2f}')
    print(f'  Diff:     {np.mean(diff):+.2f} (Wilcoxon p={p:.4f})')
    print()

print('Note: With n=7 and 1 seed, statistical power is limited.')
print('However, the effect sizes are so large (DDPM is 2-10x worse) that')
print('formal testing is unnecessary to reach a conclusion.')

---

## 8. Cross-Experiment Comparison

| Experiment | MAE (Gy) | Global Gamma | PTV Gamma | PTV70 D95 Gap | Training Time |
|------------|----------|--------------|-----------|---------------|---------------|
| baseline_v23 (seed42) | 4.80 ± 2.45 | 28.1 ± 12.6% | 85.5 ± 10.9% | -0.86 ± 0.92 Gy | ~6h |
| **baseline_v23 (3-seed agg.)** | **4.22 ± 0.53** | **33.8 ± 4.6%** | **80.2 ± 5.3%** | **-1.76 ± 0.69 Gy** | ~18h |
| combined_loss_2.5:1 (seed42) | 4.88 ± 2.09 | 27.4 ± 11.2% | 95.1 ± 3.5% | +0.12 ± 0.60 Gy | ~6h |
| **combined_loss_2.5:1 (3-seed agg.)** | **4.07 ± 0.64** | **30.4 ± 3.6%** | **94.3 ± 2.2%** | **+0.06 ± 0.26 Gy** | ~18h |
| **DDPM v2.3 (seed42)** | **9.04 ± 1.95** | **9.2 ± 2.1%** | **0.0%** | **-65.5 ± 0.8 Gy** | **33.3h** |

**Key comparisons:**
- DDPM is **2x worse on MAE** than baseline (9.04 vs 4.80 Gy)
- DDPM is **3x worse on global gamma** than baseline (9.2% vs 28.1%)
- DDPM has **0% PTV gamma** — complete clinical failure (vs 85.5% baseline, 95.1% combined)
- DDPM predicts only ~7% of target dose in PTV (D95 ~5 Gy vs ~70 Gy)
- DDPM takes **5.5x longer to train** than baseline (33.3h vs ~6h)
- The DDPM does **not** improve global gamma — the original hypothesis is rejected

---

## 9. Conclusions, Limitations, and Next Steps

### Conclusions

1. **DDPM dramatically underperforms the baseline U-Net on all metrics.** The denoising process fails to reconstruct the correct dose magnitude, predicting ~10-15% of the actual PTV dose.
2. **The hypothesis that diffusion improves global/low-dose gamma is rejected.** Global gamma is 3x worse than baseline, not better.
3. **DDPM's training instability makes it impractical.** Val MAE oscillated 5-84 Gy throughout training, making it impossible to assess convergence or select a good checkpoint with confidence.
4. **The architecture is not the bottleneck.** Both DDPM and baseline use ~23.7M parameter U-Nets; the difference is the training paradigm (diffusion vs direct regression).
5. **This confirms the earlier v2.2.0 finding** — DDPM does not outperform direct regression for VMAT dose prediction. The v2.3 data did not change this conclusion.

### Possible explanations for DDPM failure

- **Dose distributions are not natural images.** Diffusion models excel at image generation where they learn complex multi-modal distributions. Dose prediction is a well-conditioned regression problem — given anatomy and constraints, there is essentially one correct dose distribution. The generative overhead of diffusion adds noise without benefit.
- **DDIM sampling artifacts.** The 50-step DDIM denoising may not be sufficient to reconstruct the full dose magnitude, leading to systematic underprediction. The output range [0, 1.1] spans the correct scale, but dose is not concentrated in the right locations.
- **Patch-based DDIM.** Running DDIM independently per patch during inference may prevent the model from maintaining global dose consistency across the volume.

### Limitations

- **Single seed (n=1).** However, the effect size is so large (D95 error of -65 Gy) that additional seeds would not change the conclusion.
- **No hyperparameter tuning for DDPM.** The DDPM used default settings; tuning might improve it, but it would need to close a 60+ Gy gap.
- **No ensemble DDIM.** Multiple DDIM runs averaged together might improve results, but the fundamental magnitude error suggests a deeper problem.

### Next Steps

- [x] Close GitHub #49 (DDPM re-evaluation) with negative result
- [x] Update GitHub #27 (DDPM decision) with v2.3 confirmation
- [ ] Focus on dataset expansion (200+ cases) to improve combined_loss_2.5:1
- [ ] Do NOT invest further in DDPM unless a fundamentally different approach is proposed

---

## 10. Artifacts

| Artifact | Path |
|----------|------|
| Best Checkpoint | `runs/ddpm_v23_seed42/checkpoints/best-epoch=055-val/mae_gy=5.62.ckpt` |
| Training Metrics | `runs/ddpm_v23_seed42/version_1/metrics.csv` |
| Training Config | `runs/ddpm_v23_seed42/training_config.json` |
| Training Summary | `runs/ddpm_v23_seed42/training_summary.json` |
| Environment Snapshot | `runs/ddpm_v23_seed42/environment_snapshot.txt` |
| Figures (PNG + PDF) | `runs/ddpm_v23/figures/` |
| Figure Generation Script | `scripts/generate_ddpm_v23_figures.py` |
| Test Predictions | `predictions/ddpm_v23_seed42_test/prostate70gy_*_pred.npz` |
| Test Results | `predictions/ddpm_v23_seed42_test/ddpm_evaluation_results.json` |

---

*Notebook created: 2026-03-07*  
*Last updated: 2026-03-07*